In [ ]:
# Importing Libraries

In [ ]:
print("\n" + "="*80)print("IMPORTING LIBRARIES...")print("="*80)import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom scipy import statsimport warningswarnings.filterwarnings('ignore')# Configure plottingsns.set_style("whitegrid")plt.rcParams['figure.figsize'] = (12, 6)print("✓ All libraries imported successfully!")

In [ ]:
# STEP 1 : LOAD AND EXPLORE DATA

In [ ]:
print("\n" + "="*80)print("STEP 1 : LOADING AND EXPLORING DATA")print("="*80)# Load the CSV fileprint("\nLoading CSV file...")df = pd.read_csv("C:/Users/aashi/OneDrive/Desktop/NYC-311-Analysis/311_Service_Requests_from_2010_to_Present.csv")print(f"✓ Loaded {len(df):,} records with {len(df.columns)} columns")# Display basic infoprint(f"\nDataset shape: {df.shape}")print(f"\nFirst 5 rows:")print(df.head())print(f"\nColumn names:")for i, col in enumerate(df.columns, 1):    print(f"  {i}. {col}")# Check null valuesprint(f"\nNull values summary:")null_counts = df.isnull().sum()null_pct = (null_counts / len(df)) * 100null_df = pd.DataFrame({    'Column': null_counts.index,    'Null_Count': null_counts.values,    'Percentage': null_pct.values})null_df = null_df[null_df['Null_Count'] > 0].sort_values('Null_Count', ascending=False)print(null_df.to_string(index=False))

In [ ]:
# STEP 2 : DATA CLEANING

In [ ]:
print("\n" + "="*80)print("STEP 2 : DATA CLEANING")print("="*80)initial_count = len(df)# Remove null Closed Dateprint("\nRemoving records with null Closed Date...")df = df[df['Closed Date'].notna()]print(f"✓ Removed {initial_count - len(df):,} records")# Convert datesprint("\nConverting date columns...")df['Created Date'] = pd.to_datetime(df['Created Date'], errors='coerce')df['Closed Date'] = pd.to_datetime(df['Closed Date'], errors='coerce')# Remove invalid timelines (Closed before Created)print("Removing records with invalid timeline (Closed < Created)...")initial_valid = len(df)df = df[(df['Closed Date'] >= df['Created Date'])].copy()print(f"✓ Removed {initial_valid - len(df):,} records")# Calculate closing time in secondsprint("\nCalculating Request_Closing_Time...")df['Request_Closing_Time'] = (df['Closed Date'] - df['Created Date']).dt.total_seconds()df['Request_Closing_Time_Hours'] = df['Request_Closing_Time'] / 3600print(f"Average closing time: {df['Request_Closing_Time_Hours'].mean():.2f} hours")print(f"Median closing time: {df['Request_Closing_Time_Hours'].median():.2f} hours")# Impute missing citiesprint("\nHandling missing City values...")null_cities = df['City'].isnull().sum()df['City'] = df['City'].fillna('Unknown City')print(f"✓ Imputed {null_cities:,} missing city values")print(f"\n✓ Data cleaning complete!")print(f"Final dataset: {len(df):,} records")

In [ ]:
# STEP 3 : CREATING VISUALIZATIONS

In [ ]:
print("\n" + "="*80)print("STEP 2 : DATA CLEANING")print("="*80)# Visualization 1: Null values frequencyprint("\n1. Creating null values frequency plot...")plt.figure(figsize=(14, 8))null_counts_all = df.isnull().sum()null_counts_all = null_counts_all[null_counts_all > 0].sort_values(ascending=False)plt.barh(range(len(null_counts_all)), null_counts_all.values, color='coral')plt.yticks(range(len(null_counts_all)), null_counts_all.index)plt.xlabel('Number of Null Values')plt.title('Null Values Distribution')plt.tight_layout()plt.savefig('01_null_values_frequency.png', dpi=300, bbox_inches='tight')plt.close()print("✓ Saved: 01_null_values_frequency.png")# Visualization 2: Complaints by Cityprint("2. Creating complaints by city plot...")city_counts = df['City'].value_counts().head(15)plt.figure(figsize=(14, 8))city_counts.plot(kind='bar', color='steelblue')plt.title('Top 15 Cities with Most Complaints')plt.xlabel('City')plt.ylabel('Number of Complaints')plt.xticks(rotation=45, ha='right')plt.tight_layout()plt.savefig('02_complaints_by_city.png', dpi=300, bbox_inches='tight')plt.close()print("✓ Saved: 02_complaints_by_city.png")# Visualization 3: Brooklyn distributionprint("3. Creating Brooklyn spatial plots...")brooklyn_df = df[df['City'] == 'BROOKLYN']brooklyn_coords = brooklyn_df[['Latitude', 'Longitude']].dropna()if len(brooklyn_coords) > 100:    fig, axes = plt.subplots(1, 2, figsize=(16, 6))        # Scatter    axes[0].scatter(brooklyn_coords['Longitude'], brooklyn_coords['Latitude'],                   alpha=0.5, s=10, color='blue')    axes[0].set_title('Scatter: Complaints in Brooklyn')    axes[0].set_xlabel('Longitude')    axes[0].set_ylabel('Latitude')        # Hexbin    axes[1].hexbin(brooklyn_coords['Longitude'], brooklyn_coords['Latitude'],                  gridsize=30, cmap='YlOrRd', mincnt=1)    axes[1].set_title('Hexbin: Complaint Concentration in Brooklyn')    axes[1].set_xlabel('Longitude')    axes[1].set_ylabel('Latitude')        plt.tight_layout()    plt.savefig('03_brooklyn_complaint_distribution.png', dpi=300, bbox_inches='tight')    plt.close()    print("✓ Saved: 03_brooklyn_complaint_distribution.png")else:    print("⚠ Insufficient Brooklyn data for spatial plots")# Visualization 4: Complaint typesprint("4. Creating complaint types plot...")complaint_counts = df['Complaint Type'].value_counts()plt.figure(figsize=(14, 8))complaint_counts.head(15).plot(kind='bar', color='teal')plt.title('Top 15 Types of Complaints')plt.xlabel('Complaint Type')plt.ylabel('Count')plt.xticks(rotation=45, ha='right')plt.tight_layout()plt.savefig('04_complaint_types.png', dpi=300, bbox_inches='tight')plt.close()print("✓ Saved: 04_complaint_types.png")# Visualization 5: Complaints by city stackedprint("5. Creating complaints by city stacked plot...")top_cities = df['City'].value_counts().head(5).indextop_complaints = df['Complaint Type'].value_counts().head(8).indexdf_filtered = df[(df['City'].isin(top_cities)) & (df['Complaint Type'].isin(top_complaints))]city_complaint = pd.crosstab(df_filtered['City'], df_filtered['Complaint Type'])plt.figure(figsize=(14, 8))city_complaint.plot(kind='bar', figsize=(14, 8), colormap='tab20')plt.title('Complaint Distribution by City')plt.xlabel('City')plt.ylabel('Count')plt.xticks(rotation=45, ha='right')plt.legend(title='Complaint Type', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)plt.tight_layout()plt.savefig('05_complaints_by_city_stacked.png', dpi=300, bbox_inches='tight')plt.close()print("✓ Saved: 05_complaints_by_city_stacked.png")# Visualization 6: Average closing timeprint("6. Creating average closing time plot...")avg_closing = df.groupby('Complaint Type')['Request_Closing_Time_Hours'].mean().sort_values(ascending=False)plt.figure(figsize=(14, 8))avg_closing.head(15).plot(kind='barh', color='coral')plt.title('Average Closing Time by Complaint Type')plt.xlabel('Hours')plt.tight_layout()plt.savefig('06_avg_closing_time_by_complaint.png', dpi=300, bbox_inches='tight')plt.close()print("✓ Saved: 06_avg_closing_time_by_complaint.png")# Visualization 7: Response time analysis (comprehensive)print("7. Creating comprehensive response time analysis...")fig, axes = plt.subplots(2, 2, figsize=(16, 12))# Distributionaxes[0, 0].hist(df['Request_Closing_Time_Hours'], bins=50, color='skyblue', edgecolor='black')axes[0, 0].set_title('Distribution of Closing Time')axes[0, 0].set_xlabel('Hours')axes[0, 0].set_xlim(0, 500)# Box plotaxes[0, 1].boxplot(df['Request_Closing_Time_Hours'], vert=True)axes[0, 1].set_title('Box Plot of Closing Time')axes[0, 1].set_ylabel('Hours')axes[0, 1].set_ylim(0, 500)# Average by top complaintstop_10 = complaint_counts.head(10)avg_by_complaint = df[df['Complaint Type'].isin(top_10.index)].groupby('Complaint Type')['Request_Closing_Time_Hours'].mean().sort_values()axes[1, 0].barh(range(len(avg_by_complaint)), avg_by_complaint.values, color='lightgreen')axes[1, 0].set_yticks(range(len(avg_by_complaint)))axes[1, 0].set_yticklabels(avg_by_complaint.index, fontsize=9)axes[1, 0].set_title('Average Time: Top 10 Complaints')axes[1, 0].set_xlabel('Hours')# Summary statsaxes[1, 1].axis('off')stats_text = f"""SUMMARY STATISTICSMean: {df['Request_Closing_Time_Hours'].mean():.2f} hrsMedian: {df['Request_Closing_Time_Hours'].median():.2f} hrsStd Dev: {df['Request_Closing_Time_Hours'].std():.2f} hrsMin: {df['Request_Closing_Time_Hours'].min():.2f} hrsMax: {df['Request_Closing_Time_Hours'].max():.2f} hrsTotal Records: {len(df):,}Unique Complaints: {df['Complaint Type'].nunique()}Unique Cities: {df['City'].nunique()}"""axes[1, 1].text(0.1, 0.5, stats_text, fontsize=11, family='monospace',                 bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))plt.tight_layout()plt.savefig('07_response_time_analysis.png', dpi=300, bbox_inches='tight')plt.close()print("✓ Saved: 07_response_time_analysis.png")print("\n✓ All visualizations created!")

In [ ]:
# STEP 4 : STATISTICAL ANALYSIS

In [ ]:
print("\n" + "="*80)print("STEP 4 : STATISTICAL ANALYSIS")print("="*80)# Kruskal-Wallis test for complaint typesprint("\nKruskal-Wallis H Test: Complaint Types")print("-" * 80)print("H0: All complaint types have equal distribution of closing times")print("H1: At least one complaint type differs")print(f"Significance Level (α): 0.05")top_complaints_test = df['Complaint Type'].value_counts().head(10).indexgroups = []for complaint in top_complaints_test:    group_data = df[df['Complaint Type'] == complaint]['Request_Closing_Time'].dropna()    if len(group_data) > 0:        groups.append(group_data.values)h_stat, p_val = stats.kruskal(*groups)print(f"\nResults:")print(f"  H-statistic: {h_stat:.6f}")print(f"  P-value: {p_val:.10f}")if p_val < 0.05:    print(f"\n✓ REJECT H0 (p < 0.05)")    print("  Conclusion: Complaint types have SIGNIFICANTLY DIFFERENT closing times")else:    print(f"\n✓ FAIL TO REJECT H0 (p ≥ 0.05)")    print("  Conclusion: Complaint types have similar closing times")# Kruskal-Wallis test for citiesprint("\n" + "-" * 80)print("Kruskal-Wallis H Test: Cities")print("-" * 80)top_cities_test = df['City'].value_counts().head(5).indexcity_groups = []for city in top_cities_test:    city_data = df[df['City'] == city]['Request_Closing_Time'].dropna()    if len(city_data) > 0:        city_groups.append(city_data.values)h_stat_city, p_val_city = stats.kruskal(*city_groups)print(f"\nResults:")print(f"  H-statistic: {h_stat_city:.6f}")print(f"  P-value: {p_val_city:.10f}")if p_val_city < 0.05:    print(f"\n✓ REJECT H0 (p < 0.05)")    print("  Conclusion: Cities have SIGNIFICANTLY DIFFERENT closing times")else:    print(f"\n✓ FAIL TO REJECT H0 (p ≥ 0.05)")    print("  Conclusion: Cities have similar closing times")

In [ ]:
# STEP 5 : SUMMARY ANALYSIS

In [ ]:
print("\n" + "="*80)print("STEP 5 : KEY INSIGHTS")print("="*80)print("\n1. DATA QUALITY:")print(f"   - Initial records: {initial_count:,}")print(f"   - Final records: {len(df):,}")print(f"   - Records removed: {initial_count - len(df):,}")print("\n2. TOP COMPLAINT TYPES:")for i, (complaint, count) in enumerate(complaint_counts.head(5).items(), 1):    pct = (count / len(df)) * 100    print(f"   {i}. {complaint}: {count:,} ({pct:.1f}%)")print("\n3. TOP CITIES:")for i, (city, count) in enumerate(city_counts.head(5).items(), 1):    pct = (count / len(df)) * 100    print(f"   {i}. {city}: {count:,} ({pct:.1f}%)")print("\n4. CLOSING TIME INSIGHTS:")print(f"   - Average: {df['Request_Closing_Time_Hours'].mean():.2f} hours")print(f"   - Median: {df['Request_Closing_Time_Hours'].median():.2f} hours")print(f"   - Range: {df['Request_Closing_Time_Hours'].min():.2f} to {df['Request_Closing_Time_Hours'].max():.2f} hours")print("\n5. STATISTICAL SIGNIFICANCE:")print(f"   - Complaint types differ: {p_val < 0.05}")print(f"   - Cities differ: {p_val_city < 0.05}")

In [ ]:
# STEP 6 : SAVE SUMMARY

In [ ]:
print("\n" + "="*80)print("STEP 6 : SAVING SUMMARY")print("="*80)summary_data = {    'Metric': [        'Total Records',        'Unique Complaints',        'Unique Cities',        'Avg Closing Time (hours)',        'Median Closing Time (hours)',        'Max Closing Time (hours)',        'Data Start Date',        'Data End Date'    ],    'Value': [        f"{len(df):,}",        df['Complaint Type'].nunique(),        df['City'].nunique(),        f"{df['Request_Closing_Time_Hours'].mean():.2f}",        f"{df['Request_Closing_Time_Hours'].median():.2f}",        f"{df['Request_Closing_Time_Hours'].max():.2f}",        str(df['Created Date'].min().date()),        str(df['Created Date'].max().date())    ]}summary_df = pd.DataFrame(summary_data)summary_df.to_csv('analysis_summary.csv', index=False)print("\n✓ Saved: analysis_summary.csv")print("\nSummary:")print(summary_df.to_string(index=False))

In [ ]:
# COMPLETION

In [ ]:
print("\n" + "="*80)print("STep 7 : ANALYSIS COMPLETE")print("="*80)print("\nGenerated Files:")print("  - 01_null_values_frequency.png")print("  - 02_complaints_by_city.png")print("  - 03_brooklyn_complaint_distribution.png")print("  - 04_complaint_types.png")print("  - 05_complaints_by_city_stacked.png")print("  - 06_avg_closing_time_by_complaint.png")print("  - 07_response_time_analysis.png")print("  - analysis_summary.csv")